## Dependecies and Imports

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q -U accelerate==0.27.2
!pip install -q -U transformers==4.37.2
!pip install -q -U datasets==2.17.0
!pip install -q -U evaluate==0.4.1

In [ ]:
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, hamming_loss, recall_score, precision_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from torch.utils.data import DataLoader
from datasets import Dataset, DatasetDict
from tqdm.auto import tqdm
import ast

## Necessary Functions

In [ ]:
# Multi-Label Classification Evaluation Metrics
def multi_labels_metrics(predictions, label_values, threshold=0.5):
    sigmoid = torch.nn.Sigmoid()
    # apply sigmoid on predictions to convert values between 0 and 1
    probs = sigmoid(torch.Tensor(predictions))

    # using threshold to turn them into integer predictions
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs>=threshold)] = 1
    y_true = label_values

    f1 = f1_score(y_true, y_pred, average = 'samples')
    precision = precision_score(y_true, y_pred, average = 'samples')
    recall = recall_score(y_true, y_pred, average = 'samples')
    hamming = hamming_loss(y_true, y_pred)

    metrics = {
      "hamming_loss": hamming,
      "f1": f1,
      "recall": recall,
      "precision": precision
    }

    return metrics

def preprocess_function(example):
    text = example['text']
    related_labels = ast.literal_eval(example['related_labels'])
    # a list of empty labels
    temp_labels = [0. for i in range(len(labels))]

    for rl in related_labels:
        label_id = label2id[rl]
        temp_labels[label_id] = 1.

    example = tokenizer(text, padding='max_length', truncation=True)
    example['labels'] = temp_labels
    return example

#get number of model parameters
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params/1e9:.3f} B\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

## Load Model and Dataset

In [ ]:
model_path = 'AdnanSadi/BioDisSumBert_DDXPlus_2'

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast= True)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

print('Device - ', device)
print(print_number_of_trainable_model_parameters(model))

In [ ]:
# load csv files
valid_data = pd.read_csv('/content/drive/MyDrive/Differential Diagnosis Project/Dataset/Sampled DDXPlus for Classification/val/validation_set.csv')
test_data = pd.read_csv('/content/drive/MyDrive/Differential Diagnosis Project/Dataset/Sampled DDXPlus for Classification/test/test_set.csv')
test_data_custom = pd.read_csv('/content/drive/MyDrive/Differential Diagnosis Project/Dataset/Sampled DDXPlus for Classification/test/with_modifications/custom_test_set_100_patients.csv')
test_data_typos = pd.read_csv('/content/drive/MyDrive/Differential Diagnosis Project/Dataset/Sampled DDXPlus for Classification/test/with_modifications/test_set_with_typos.csv')
test_data_text_rmv = pd.read_csv('/content/drive/MyDrive/Differential Diagnosis Project/Dataset/Sampled DDXPlus for Classification/test/with_modifications/test_set_with_text_removal.csv')

# dataset dictionary
dataset = DatasetDict()
valid_dataset = Dataset.from_pandas(valid_data)
test_dataset = Dataset.from_pandas(test_data)
test_dataset_custom = Dataset.from_pandas(test_data_custom)
test_dataset_typos = Dataset.from_pandas(test_data_typos)
test_dataset_text_rmv = Dataset.from_pandas(test_data_text_rmv)


dataset['validation'] = valid_dataset
dataset['test'] = test_dataset
dataset['test_custom'] = test_dataset_custom
dataset['test_typos'] = test_dataset_typos
dataset['test_text_rmv'] = test_dataset_text_rmv
print('\n', dataset)

In [ ]:
#define labels
labels = ast.literal_eval(test_data['all_labels'][0])
label2id = {label:idx for idx, label in enumerate(labels)}
id2label = {idx:label for label, idx in label2id.items()}

# tokenize dataset
tokenized_dataset = dataset.map(preprocess_function, remove_columns=dataset["test"].column_names)
columns = tokenized_dataset['test'].column_names
tokenized_dataset.set_format(type='torch', columns=columns)
print('\n', tokenized_dataset)

In [ ]:
# define dataloaders
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

batch_size = 16

valid_dataloader = DataLoader(
       tokenized_dataset["validation"], shuffle=True, batch_size=batch_size, collate_fn=data_collator
    )
test_dataloader = DataLoader(
       tokenized_dataset["test"], shuffle=True, batch_size=batch_size, collate_fn=data_collator
    )
test_custom_dataloader = DataLoader(
       tokenized_dataset["test_custom"], shuffle=True, batch_size=batch_size, collate_fn=data_collator
)
test_typos_dataloader = DataLoader(
       tokenized_dataset["test_typos"], shuffle=True, batch_size=batch_size, collate_fn=data_collator
)
test_text_rmv_dataloader = DataLoader(
       tokenized_dataset["test_text_rmv"], shuffle=True, batch_size=batch_size, collate_fn=data_collator
)

## Test Loop for Multi-label Metrics

In [ ]:
def test_loop(eval_dataloader, model, threshold):
    all_preds = torch.tensor([]).to(device)
    all_labels = torch.tensor([]).to(device)
    model.eval()

    for batch in tqdm(eval_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.no_grad():
            outputs = model(**batch)
            preds = outputs.logits.squeeze()
        
        # Check if preds is 1D and reshape (for when last batch has only 1 sample)
        if preds.dim() == 1:
            preds = preds.unsqueeze(0)  # Add a dimension to make it 2D

        all_preds = torch.cat((all_preds, preds))
        all_labels = torch.cat((all_labels, batch['labels']))

    metric_scores = multi_labels_metrics(predictions=all_preds.cpu(), label_values=all_labels.cpu(), threshold=threshold)

    print(f"""F1 - {round(metric_scores['f1'], 4)}
precision - {round(metric_scores['precision'], 4)}
recall - {round(metric_scores['recall'], 4)}
Hamming_Loss - {round(metric_scores['hamming_loss'], 4)}""")

In [ ]:
# validation set results
test_loop(valid_dataloader, model, 0.5)

In [ ]:
# hold out test set results
test_loop(test_dataloader, model, 0.5)

In [ ]:
# hold out test set with typos results
test_loop(test_typos_dataloader, model, 0.5)

In [ ]:
# hold out test set with text removal results
test_loop(test_text_rmv_dataloader, model, 0.5)

In [ ]:
# custom test set results
test_loop(test_custom_dataloader, model, 0.2)

## Test Loop for Differential Diagnosis Metrics

In [ ]:
def calculate_gtda(differential_diag, gt_diag):
    '''
    Function for getting the GTD accuracy score

    Args:
    differential_diag: a python list containing the list names of the diseases in differential diagnosis for each sample. The
                       length of the list should be same as the length of the test set.
    gt_diag: a python list containing the name of the ground truth diagnosis for each sample. The length of the list should be
             same as the length of the test set.

    Return:
    gtda_score: returns the ground truth in differential accuracy score
    '''
    num_correct = 0

    for dd, gt in zip(differential_diag, gt_diag):
        if gt in dd:
            num_correct+= 1
        else:
            continue

    gtda_score = num_correct/len(gt_diag)
    return gtda_score

In [ ]:
# testing out the calculate_gtda() method
test_dd = [['Pulmonary embolism', 'Spontaneous pneumothorax', 'Possible NSTEMI / STEMI', 'Panic attack', 'Pericarditis',
      'Guillain-Barré syndrome', 'Atrial fibrillation', 'GERD', 'Acute dystonic reactions',
      'Myasthenia gravis', 'Anemia', 'Myocarditis', 'Scombroid food poisoning', 'PSVT', 'Stable angina'],
       ['GERD', 'Anaphylaxis'],
       ['Possible NSTEMI / STEMI', 'Pulmonary embolism', 'Viral pharyngitis', 'Unstable angina', 'Stable angina']]
test_gt = ['Spontaneous pneumothorax', 'Anemia', 'Anaphylaxis']

print(calculate_gtda(test_dd, test_gt))

In [ ]:
def get_differentials(test_dataset, threshold=0.5, top_5=False):
  """
  This function runs inference on the given test set. It returns the predicted differential diagnosis and ground truth pathology, which can then be used to calculate the GTD score.
  args:
        test_dataset: a dataframe containing test data.
        threshold: prediction confidence threshold (between 0 and 1) for calculating GTD score. Default value is 0.5.
        top_5: a boolean variable. Setting it to True will return the top-5 differential diagnoses for each sample. In this case, the threshold is not used.

    return:
        all_dds: a python list containing the list names of the diseases in differential diagnosis for each sample. The length of the list should be same as the length of the test set.
        all_gts: a python list containing the name of the ground truth pathology for each sample.
  """
  all_dds = []
  all_gts = []

  for i in tqdm(range(len(test_dataset))):
    text = test_dataset['text'][i]
    encoding = tokenizer(text, padding='max_length', truncation=True, return_tensors="pt")
    encoding = {k: v.to(device) for k,v in encoding.items()}

    outputs = model(**encoding)
    logits = outputs.logits

    # apply sigmoid + threshold
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(logits.squeeze().cpu())

    # get the top-5 predictions
    if top_5 == True:
      probs_np = probs.detach().numpy()
      top_5_indices = np.argsort(-probs_np)[:5]  # Get indices of top five probabilities, negate the array for descending order
      top_5_preds = np.zeros_like(probs_np)
      top_5_preds[top_5_indices] = 1 # Set index values to 1, where top five probabilities exist

      # turn predicted id's into actual label names
      top_5_dds = [id2label[idx] for idx, label in enumerate(top_5_preds) if label == 1.0]

      all_dds.append(top_5_dds)
      all_gts.append(test_dataset['pathology'][i])

    # get predictions where the probability values larger than the threshold
    else:
      threshold = threshold
      predictions = np.zeros(probs.shape)
      predictions[np.where(probs >= threshold)] = 1

      # turn predicted id's into actual label names
      predicted_dds = [id2label[idx] for idx, label in enumerate(predictions) if label == 1.0]
      all_dds.append(predicted_dds)
      all_gts.append(test_dataset['pathology'][i])

  return all_dds, all_gts

In [ ]:
# pass the appropriate dataframe in the get_differential() method. For example: test_data, test_data_typos, test_data_custom, test_data_text_rmv
all_dds, all_gts = get_differentials(test_data_custom, threshold=0.2, top_5=False)
print("GTDA score: ", round(calculate_gtda(all_dds, all_gts), 4))

In [ ]:
# Sanity Checks
idx = 11
print(len(all_dds), len(all_gts))
print(all_dds[idx])
print(all_gts[idx])

### Calculating GTDA for thresholds 0.5, 0.95 and top-5 together (Saving Time)

In [ ]:
df = test_data_custom # test_data, test_data_typos, test_data_custom, test_data_text_rmv

In [ ]:
all_dds = []
all_dds_at_k = []
all_gts = []
all_top_5_dds = []

# getting differentials at thresholds 0.5, 0.95, and top-5.
for i in tqdm(range(len(df))):
    text = df['text'][i]
    encoding = tokenizer(text, padding='max_length', truncation=True, return_tensors="pt")
    encoding = {k: v.to(device) for k,v in encoding.items()}

    outputs = model(**encoding)
    logits = outputs.logits

    # apply sigmoid + threshold
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(logits.squeeze().cpu())

    # get predictions where the probability values larger than the threshold
    threshold = 0.5
    predictions = np.zeros(probs.shape)
    predictions[np.where(probs >= threshold)] = 1

    # get predictions the probability values larger than threshold_k
    threshold_k = 0.95
    predictions_at_k = np.zeros(probs.shape)
    predictions_at_k[np.where(probs >= threshold_k)] = 1

    # get the top-5 probability values for the predictions
    probs_np = probs.detach().numpy()
    top_5_indices = np.argsort(-probs_np)[:5]  # Get indices of top five probabilities, negate the array for descending order
    top_5_preds = np.zeros_like(probs_np)
    top_5_preds[top_5_indices] = 1 # Set index values to 1, where top five probabilities exist

    # turn predicted id's into actual label names
    predicted_dds = [id2label[idx] for idx, label in enumerate(predictions) if label == 1.0]
    predicted_dds_at_k = [id2label[idx] for idx, label in enumerate(predictions_at_k) if label == 1.0]
    top_5_dds = [id2label[idx] for idx, label in enumerate(top_5_preds) if label == 1.0]

    all_dds.append(predicted_dds)
    all_dds_at_k.append(predicted_dds_at_k)
    all_top_5_dds.append(top_5_dds)
    all_gts.append(df['pathology'][i])

In [ ]:
print("GTDA score: ", round(calculate_gtda(all_dds, all_gts), 4))
print("GTDA@0.95 score:", round(calculate_gtda(all_dds_at_k, all_gts), 4))
print("GTDA@top-5 score:", round(calculate_gtda(all_top_5_dds, all_gts), 4))